# Senyas: train the FSL sign model (free, on Google Colab)

1. In Google Drive, make a folder `Senyas` and upload into it:
   - the `ml` folder from the project
   - the FSL-105 zip from https://data.mendeley.com/datasets/48y2y99mb9/2 (CC BY 4.0)
   - optional: Sign Lab exports (`senyas_fsl_recordings_*.zip`)
2. Run the cells in order (Runtime → Run all). The CPU runtime is enough.
3. Download `fsl_model.tflite`, `fsl_labels.txt` and `fsl_model.json` from `senyas/ml/classifier/out` and put them in `android/app/src/main/assets`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Senyas/ml
!pip install -q -r requirements.txt

In [ ]:
# Unzip FSL-105 next to the ml folder (once). Change the zip name if yours differs.
import glob, os
zips = glob.glob('/content/drive/MyDrive/Senyas/*FSL*105*.zip') + glob.glob('/content/drive/MyDrive/Senyas/48y2y99mb9*.zip')
print(zips)
if zips and not os.path.isdir('/content/FSL-105'):
    !unzip -q "{zips[0]}" -d /content/FSL-105
!find /content/FSL-105 -maxdepth 3 -name '*.csv'

In [ ]:
# Point these at the CSV files printed above.
ROOT = '/content/FSL-105'
TRAIN_CSV = glob.glob(ROOT + '/**/train.csv', recursive=True)[0]
TEST_CSV = glob.glob(ROOT + '/**/test.csv', recursive=True)[0]
LABELS_CSV = glob.glob(ROOT + '/**/labels.csv', recursive=True)[0]
CLIPS_ROOT = os.path.dirname(TRAIN_CSV)
print(TRAIN_CSV, TEST_CSV, LABELS_CSV)

In [ ]:
# Videos -> hand landmarks (the slow step; finished clips are skipped if you re-run).
!python extract_landmarks.py --csv "$TRAIN_CSV" --root "$CLIPS_ROOT" --labels "$LABELS_CSV" --out data/fsl105/train --every 2
!python extract_landmarks.py --csv "$TEST_CSV" --root "$CLIPS_ROOT" --labels "$LABELS_CSV" --out data/fsl105/test --every 2

In [ ]:
# Optional: add recordings exported from the app's Sign Lab.
for z in glob.glob('/content/drive/MyDrive/Senyas/senyas_fsl_recordings_*.zip'):
    !unzip -q -o "{z}" -d data/phone
extra = 'data/phone' if os.path.isdir('data/phone') else ''
!python train.py --data data/fsl105/train $extra --test-data data/fsl105/test --out out

In [ ]:
print(open('out/report.txt').read())
from google.colab import files
for name in ('fsl_model.tflite', 'fsl_labels.txt', 'fsl_model.json'):
    files.download('out/' + name)